# EDA и журнал исследований: прогноз дохода клиента

Этот ноутбук — единая точка правды по данным и всем проверенным
гипотезам проекта. Он заменяет прежние отдельные EDA, baseline notebook
и набор markdown-отчётов.

Принцип проверки: первичный screening делается на random outer split,
а решение о production — только после переноса с зафиксированными
параметрами на временной holdout (январь–май → июнь). Отрицательная
`delta_wmae` означает улучшение.

Production-код намеренно находится не здесь, а в двух entry point:
`train_full_champion.py` и `train_compact_champion.py`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CATEGORICAL_COLS, CSV_READ_KWARGS, DATE_COL, FIGURES_DIR,
    ID_COL, PARTIAL_OUTPUTS_DIR, TARGET_COL, TEST_PATH, TRAIN_PATH,
    WEIGHT_COL,
)
from src.feature_engineering import (
    GROUP_BUILDERS, INCOME_ANCHORS, add_feature_groups,
)
from src.ordinal_routing import DEFAULT_BAND_EDGES, income_band
from src.preprocessing import preprocess
from src.wmae_distribution import recover_weight_rule

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
sns.set_theme(style="whitegrid")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## 1. Данные, схема и временная ось


In [ ]:
train_raw = pd.read_csv(TRAIN_PATH, **CSV_READ_KWARGS, low_memory=False)
test_raw = pd.read_csv(TEST_PATH, **CSV_READ_KWARGS, low_memory=False)
train, converted_train = preprocess(train_raw, is_train=True)
test, converted_test = preprocess(test_raw, is_train=False)

schema = pd.DataFrame({
    "rows": [len(train), len(test)],
    "columns": [train.shape[1], test.shape[1]],
    "date_min": [train[DATE_COL].min(), test[DATE_COL].min()],
    "date_max": [train[DATE_COL].max(), test[DATE_COL].max()],
    "converted_numeric": [len(converted_train), len(converted_test)],
}, index=["train", "test"])
display(schema)

train_features = set(train.columns) - {TARGET_COL, WEIGHT_COL}
test_features = set(test.columns)
assert train_features == test_features, "train/test feature schemas differ"
print("Feature schema совпадает; target leakage columns отсутствуют в test.")


`dt` создаёт реальный временной сдвиг: train заканчивается июнем, test
начинается июлем. Поэтому random CV недостаточно. Любой небольшой плюс,
исчезающий на June holdout, считается p-mining и не попадает в production.


## 2. Target, длинный хвост и точная природа веса WMAE


In [ ]:
target = train[TARGET_COL].to_numpy(float)
weights = train[WEIGHT_COL].to_numpy(float)
target_summary = train[TARGET_COL].describe(
    percentiles=[.01, .05, .25, .5, .75, .9, .95, .99, .999]
).to_frame("target")
display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train[TARGET_COL], bins=100, ax=axes[0])
axes[0].set_title("Target: длинный правый хвост")
sns.histplot(np.log1p(train[TARGET_COL]), bins=100, ax=axes[1])
axes[1].set_title("log1p(target)")
plt.tight_layout()
plt.show()


In [ ]:
weight_rule = recover_weight_rule(target, weights)
display(pd.Series(weight_rule, name="recovered WMAE rule"))

reconstructed = np.minimum(
    np.abs(target / weight_rule["center"] - 1.0), weight_rule["cap"]
)
assert np.max(np.abs(reconstructed - weights)) < 1e-8

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(target, weights, s=5, alpha=.2)
ax.set(xlabel="target", ylabel="w", title="Вес является точной функцией target")
plt.show()


Ключевой вывод: `w(y) = min(|y / 84017.08 - 1|, 2.5707)`. Поэтому
оптимальный point prediction для WMAE — не обычная conditional median,
а медиана распределения `f(y|x) · w(y)`. Именно это объясняет устойчивый
независимый сигнал exact-WMAE distribution head в полном чемпионе.


## 3. Пропуски, источники дохода и дефицит информации


In [ ]:
feature_columns = [c for c in train.columns if c not in {ID_COL, TARGET_COL, WEIGHT_COL}]
missing = pd.DataFrame({
    "train_missing": train[feature_columns].isna().mean(),
    "test_missing": test[feature_columns].isna().mean(),
})
missing["shift"] = missing["test_missing"] - missing["train_missing"]
display(missing.sort_values("shift", key=abs, ascending=False).head(25))

anchor_coverage = train[INCOME_ANCHORS].notna().sum(axis=1)
coverage_table = (
    pd.DataFrame({"anchors": anchor_coverage, "target": target, "w": weights})
    .groupby("anchors")
    .agg(rows=("target", "size"), target_median=("target", "median"),
         weight_sum=("w", "sum"))
)
coverage_table["weight_share"] = coverage_table["weight_sum"] / weights.sum()
display(coverage_table)


Историческая outer-fold диагностика:

| Режим | Доля веса | Локальный WMAE |
|---|---:|---:|
| `salary_6to12m_avg` отсутствует | 79.3% | 70 051 |
| `salary_6to12m_avg` присутствует | 20.7% | 34 950 |
| нет income anchors | 14.6% | 72 021 |
| ровно один anchor | 49.5% | 71 643 |
| три anchors | 19.4% | 46 075 |

Модель чаще ошибается не из-за недостаточной мощности, а из-за
неидентифицируемости: у большинства клиентов есть только один шумный
proxy. Явный missing flag важен семантически (`нет источника` ≠ `0`),
но отдельная массовая генерация флагов не дала прироста CatBoost.


## 4. Каталог проверенных feature engineering гипотез


In [ ]:
feature_catalog = pd.DataFrame([
    ("anchors", "count/mean/median/std/min/max/CV/range независимых income anchors", "screening +; temporal drift"),
    ("scale", "debt/income, utilization, credit-limit shares", "дублирует anchors/raw split"),
    ("flows", "credit/debit ratio и signed balance", "нестабильно по folds"),
    ("trends", "recent/long: 3/6/12m, salary 1y/2y", "не перенеслось во времени"),
    ("regional_norm", "anchor / региональный benchmark", "слабее anchors"),
    ("expense_shares", "категория / общий spend", "ухудшение"),
    ("log_rank", "signed log1p и percentile rank", "единственный устойчивый слабый плюс"),
    ("missing_flags", "явные индикаторы отсутствия", "нейтрально для CatBoost"),
    ("recency", "dt - period_last_act_ad", "standalone +, blend -"),
], columns=["group", "construction", "validated conclusion"])
display(feature_catalog)

engineered_preview = add_feature_groups(
    pd.concat([train.head(200), test.head(200)], ignore_index=True),
    list(GROUP_BUILDERS),
)
new_columns = [c for c in engineered_preview if c not in train.columns]
print(f"Feature builders executable: {len(new_columns)} generated columns")
display(pd.Series(new_columns, name="generated feature").to_frame().head(80))


### 4.1 Первичный screening и проверка устойчивости


In [ ]:
screening = pd.DataFrame([
    ("baseline", 62842, 0, "reference"),
    ("anchors", 62542, -300, "перспективно только random"),
    ("scale", 62785, -57, "нет инкремента поверх anchors"),
    ("flows", 62556, -286, "проверить folds"),
    ("trends", 62624, -218, "проверить temporal"),
    ("regional_norm", 62716, -126, "слабее anchors"),
    ("expense_shares", 62924, 82, "reject"),
    ("log_rank", 62586, -256, "проверить folds"),
    ("missing_flags", 62849, 7, "neutral"),
    ("Bayesian region TE", 62768, -74, "effect absent"),
], columns=["feature_group", "wmae_fold0", "delta_wmae", "decision"])
display(screening.style.bar(subset=["delta_wmae"], align="zero"))


In [ ]:
stability = pd.DataFrame([
    ("fold0", -286, -256), ("fold1", -229, -284),
    ("fold2", -28, -29), ("fold3", 85, 66),
    ("fold4", 304, 41), ("random mean", -31, -92),
    ("temporal", 1, -71),
], columns=["split", "flows_delta", "log_rank_delta"])
display(stability)
stability.set_index("split").plot(kind="bar", figsize=(10, 4), ax=plt.gca())
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Δ WMAE; ниже лучше")
plt.title("Устойчивость feature families")
plt.show()


Итоги feature research:

- `log_rank`: средний random `−92`, temporal `−71`; слабый, но
  единственный устойчивый ручной блок.
- `flows`: средний random `−31`, temporal `+1`; отвергнут.
- `anchors + flows`: `−536/−606/−66` на первых folds, но `+518`
  temporal; найден drift доступности/качества источников.
- `trends`: `+62` temporal; отвергнут.
- `recency`: standalone distribution head `61 781 → 61 731`, но лучший
  blend `59 289 → 59 302`; не включён.
- В production LGBM router/expert используется контролируемый набор
  `anchors, scale, flows, trends, log_rank` как совместное представление;
  нельзя интерпретировать слабую одиночную абляцию как самостоятельный
  гарантированный прирост.


## 5. Что реально разделяет G0 от G2/G3


In [ ]:
capacity_signal = pd.DataFrame([
    ("hdb_bki_total_max_limit", 283, 642, 989, .711),
    ("avg_credit_turn_rur", 40, 83, 108, .649),
    ("turn_cur_cr_max_v2", 34, 71, 100, .656),
], columns=["feature", "median_G0_k", "median_G2_k", "median_G3_k", "AUC_G0_vs_G2G3"])
display(capacity_signal)


Чистые временные ratios на anchor-sparse клиентах дали AUC лишь
`0.51–0.55`. Абсолютная финансовая ёмкость сильнее. Практический вывод:
время полезно как устойчивость при известном масштабе; нормированный
тренд сам по себе выбрасывает главное различие G0/G2/G3.


## 6. Доходные группы: confusion matrix, bias и вклад в WMAE


In [ ]:
band_names = ["20–50", "50–75", "75–100", "100–150", "150–250", "250–400", "400–700", "700k+"]
random_confusion = np.array([
    [90.0,4.7,0.0,1.9,2.8,.5,0,0],
    [55.6,32.5,.7,4.9,5.0,1.2,.1,0],
    [37.9,17.5,12.8,19.0,9.8,2.6,.3,.1],
    [25.7,5.4,1.9,40.6,21.2,4.0,.8,.2],
    [16.6,2.4,.3,13.4,52.5,12.1,2.2,.5],
    [9.5,1.4,0,3.1,26.2,51.3,6.8,1.8],
    [8.2,.9,0,3.0,11.3,30.3,37.2,9.1],
    [2.9,0,0,1.0,11.4,18.1,17.1,49.5],
])
temporal_confusion = np.array([
    [90.2,4.5,.2,1.8,2.8,.5,.1,0],
    [52.9,37.3,.8,3.9,4.2,.8,.1,0],
    [36.8,16.8,13.8,21.0,9.1,2.2,.2,.1],
    [27.4,5.3,2.4,38.6,21.5,4.1,.5,.1],
    [18.9,1.9,.1,13.0,51.5,12.0,1.9,.8],
    [7.2,1.8,.2,5.2,30.5,46.9,7.1,1.1],
    [6.6,.4,0,3.5,10.2,37.2,30.1,11.9],
    [4.9,0,0,0,9.8,13.7,14.7,56.9],
])
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, matrix, title in zip(axes, [random_confusion, temporal_confusion], ["Random", "Temporal"]):
    sns.heatmap(matrix, annot=True, fmt=".1f", cmap="Blues", ax=ax,
                xticklabels=band_names, yticklabels=band_names)
    ax.set(xlabel="predicted", ylabel="true", title=f"{title}: weighted row %")
plt.tight_layout()
plt.show()


In [ ]:
group_metrics = pd.DataFrame([
    ("random","20–50",37.17,90.0,15.1,17.2,6.41),
    ("random","50–75",12.24,32.5,4.2,21.8,2.66),
    ("random","75–100",1.82,12.8,1.6,34.8,.63),
    ("random","100–150",9.43,40.6,-2.8,43.7,4.12),
    ("random","150–250",16.29,52.5,-15.4,60.1,9.79),
    ("random","250–400",13.17,51.3,-48.6,90.8,11.96),
    ("random","400–700",6.80,37.2,-104.8,168.5,11.45),
    ("random","700k+",3.09,49.5,-342.1,403.9,12.47),
    ("temporal","20–50",37.72,90.2,15.4,17.6,6.62),
    ("temporal","50–75",12.43,37.3,3.6,19.9,2.47),
    ("temporal","75–100",1.93,13.8,3.5,35.8,.69),
    ("temporal","100–150",9.83,38.6,-.7,44.2,4.34),
    ("temporal","150–250",16.13,51.5,-11.1,62.7,10.12),
    ("temporal","250–400",12.63,46.9,-47.3,89.2,11.28),
    ("temporal","400–700",6.42,30.1,-105.9,178.6,11.48),
    ("temporal","700k+",2.90,56.9,-334.1,373.2,10.82),
], columns=["split","true_band","weight_pct","exact_pct","mean_bias_k","mae_k","wmae_contribution_k"])
display(group_metrics)


Главные ошибки router: G1→G0 `55.6%`, G2→G0 `37.9%`, G6→G5
`30.3%` random (`37.2%` temporal). Но главный численный bottleneck всё
ещё 700k+: около 3% веса создают 10.8–12.5k WMAE из-за bias примерно
`−340k`. G0, напротив, массово переоценивается примерно на `+15k`.

Срезы промахов подтверждают missing-anchor режим:

- G1→G0: target median 56.5k, base 49.8k, 85.7% имеют ≤1 anchor;
- G2→G0: target median 90k, base 60.8k, bias −23.2k, 94.7% имеют ≤1 anchor;
- G6→G5: target median 460.7k, base 330k, bias −166k, 65.7% имеют ≤1 anchor.


## 7. Кластеризация и classification repair: почему не спасли


In [ ]:
clustering = pd.DataFrame([
    (8,44.46,67.88,.066,102212),
    (16,44.59,66.94,.066,97470),
    (32,45.82,68.31,.070,95337),
], columns=["clusters","exact_pct","within_1_pct","NMI","cluster_median_WMAE"])
display(clustering)


PCA + MiniBatchKMeans группирует активность и паттерны пропусков, а не
target (NMI ≈ 0.07). Кластеризация не создаёт недостающий сигнал.

Spearman/evidence-aware boundary repair чуть улучшал AUC 50k
(`0.6918 → 0.7010` random, `0.6960 → 0.7114` temporal), но ухудшал WMAE:
G1→G0 `55.59 → 55.98%`, G2→G0 `37.86 → 37.99%`. Только repair 400k
дал random `−61.6`, затем temporal `+121.2`; отвергнут. AUC ранжирует
пары, а WMAE требует правильной калибровки и правильного специалиста.


## 8. Полный журнал модельных и статистических гипотез


In [ ]:
experiments = pd.DataFrame([
    ("Power-target + salary anchor", 62792, 60874, "base for routing", "accepted"),
    ("Residual CatBoost", 63120, np.nan, "same features model noise", "rejected"),
    ("Tail classifiers + specialists", 61357, 60379, "independent tail signal", "accepted layer"),
    ("Leaf-space neighbours", 62670, 60874, "best blend only random", "rejected"),
    ("Distribution q50 blend", 62353, 60587, "small stable before stronger ensemble", "superseded"),
    ("Source expert anchors>=3", 62248, 60375, "stable small signal", "accepted layer"),
    ("LightGBM diversity", 62471, 60017, "useful before ordinal", "superseded"),
    ("Cumulative ordinal ensemble", 59830, 58023, "strong range signal", "superseded"),
    ("Hierarchical router, adaptive T", 59567, 57745, "23-model official", "accepted backbone"),
    ("Cost-sensitive boundaries", 59784, 57755, "temporal regression", "rejected"),
    ("Regret-sensitive boundaries", 59764, 57743, "~5 worse minimax", "rejected"),
    ("Trust/abstention gate", 59564, 57744, "near-zero utility", "rejected"),
    ("Shared router", 59641, 57901, "upper-tail temporal bias", "rejected"),
    ("Shared band expert + adaptive T", 59501, 57827, "16 models", "compact champion"),
    ("G0 escape classifier", 59500, 57832, "AUC .85 but bad conditional calibration", "rejected"),
    ("Exact-WMAE distribution head", 59291, 57740, "metric-aware independent signal", "full champion"),
], columns=["experiment","random_wmae","temporal_wmae","finding","decision"])
display(experiments)


Дополнительные locked результаты:

- Tail threshold AUC: 150k `.946`, 300k `.939`, 500k `.942`;
  oracle gates дают около `−9–10k`, реальные soft gates существенно меньше.
- Quantile oracle среди q10…q90 ≈ 26k WMAE, но classifier не умеет
  выбирать нужный quantile конкретной строки.
- Cost-sensitive mild: random `−169`, temporal `+53`; не переносится.
- Regret posterior: random `−80`, temporal `−2`, но полный minimax хуже
  production примерно на 5.
- Trust gate имеет weighted AUC дальнего промаха около `.81`, однако
  почти всегда WMAE-оптимально оставить correction включённым.
- Attractor escape G0/G1 vs G2/G3 имеет AUC `.8487/.8498`, но locked
  temporal ухудшает на 5.27. Хороший global rank не гарантирует
  calibration внутри уже ошибочно выбранной G0-ветви.
- Leaky clipping и skip-connection ухудшили результат. Пики диапазонов
  не являются основной причиной ошибки.


## 9. Внешнее публичное решение: найденная связь, а не копирование


Train/test публичного `botslivehere/income-Prediction-Bank` побайтово
совпали с локальными (SHA-256 train начинается `c2f9bf95…`, test
`66337d0d…`). Чужой `joblib` сознательно не десериализовался.

Большинство feature semantics у нас уже было: anchors, ratios, flows,
trends, regional normalization, expense shares, log/rank, missing flags.
Новым оказался не «секретный признак», а distribution stacking:

1. raw-MAE/log-MAE/MSE base components и их spread;
2. rolling OOT и nested OOF;
3. causal target CDF и geography shrinkage;
4. residual distribution experts;
5. residual ordinal CDF и 7 quantiles;
6. exact weighted median под восстановленную WMAE-функцию.

Минимальная честная голова q20/q50/q80 дала:

| Основа | alpha | Random | Temporal |
|---|---:|---:|---:|
| Compact | .15 | 59 501 → **59 289** | 57 827 → **57 795** |
| Full hierarchy | .20 | 59 567 → **59 291** | 57 745 → **57 740** |

Для production выбран второй вариант из-за temporal-запаса. Полный
внешний pipeline содержит около 34–36 inference-моделей; наш чемпион
переносит только доказанный metric-aware принцип и остаётся на 26.


## 10. Две production-системы


### Full champion — 26 моделей, 59 291 / 57 740

- 1 base CatBoost;
- 3 tail classifiers + 3 tail experts;
- 1 multi-anchor expert;
- 7 hierarchical LightGBM nodes;
- 8 band experts;
- 3 unweighted log-quantile LightGBM;
- adaptive temperature: `T=.3` для top G0/G4, иначе `.5`;
- final: `0.80 × hierarchical ensemble + 0.20 × exact-WMAE head`.

### Compact champion — 16 моделей, 59 501 / 57 827

- те же base/tail/source/router = 15 моделей;
- 1 band-conditioned shared expert вместо 8 отдельных;
- та же adaptive temperature;
- без distribution head.

Разница между compact и прежним 23-model backbone статистически не
отличалась от нуля по paired bootstrap, но строгая zero-margin
non-inferiority не доказана. Поэтому обе версии сохранены.


## 11. Итоговые выводы и следующий исследовательский приоритет


1. Массовое ручное FE почти исчерпано: CatBoost уже извлекает splits из
   исходных сумм и пропусков. `log_rank` — единственный слабый устойчивый
   кандидат; anchors требуют time-aware обращения.
2. Низ классифицируется плохо, но его жёсткое исправление не улучшает
   WMAE. G0/G1/G2 перекрываются именно в режиме 0–1 anchors.
3. Верхний хвост остаётся крупнейшим денежным источником ошибки: 700k+
   создаёт около пятой части общей WMAE при 3% веса.
4. Residual stacking на тех же признаках генерирует шум. Нужен новый
   независимый сигнал или честные OOF distribution meta-features.
5. Следующий крупный шаг: rolling OOT base components → component spread
   → causal target CDF как meta-features → residual distribution expert.
   Каждый слой принимать только после random + locked temporal абляции.
